# Parametric equilibrium descriptors

Derive convention-aware global descriptors from the repository GEQDSK sample and verify that its ODS representation gives the same result. Values use SI units and retain derivation metadata.

In [ ]:
import numpy as np
from vaft.data.resources import sample_geqdsk
from vaft.process.equilibrium import as_equilibrium, derive_global_descriptors

gfile = sample_geqdsk()
geq = as_equilibrium(gfile, convention=11)
ods_eq = as_equilibrium(gfile.to_omas(), convention=11)
geq_desc = derive_global_descriptors(geq)
ods_desc = derive_global_descriptors(ods_eq)
for name in ("ip", "bt0", "major_radius", "minor_radius", "elongation", "volume", "q95"):
    left, right = geq_desc[name], ods_desc[name]
    if left.available and right.available:
        np.testing.assert_allclose(left.value, right.value, rtol=1e-8, atol=1e-10)
    print(f"{name:28s} {left.value!s:>14s} {left.unit:>10s}  [{left.provenance.method}]")
print("COCOS evidence:", geq.convention)

ip                               77040.1398          A  [source]
bt0                            0.1498058775          T  [source]
major_radius                    0.429970048          m  [closed-polygon geometry]
minor_radius                    0.325967054          m  [closed-polygon geometry]
elongation                   1.355007202660426          1  [closed-polygon geometry]
volume                       1.0819146913987394        m^3  [closed-polygon geometry]
q95                          8.838592261999997          1  [linear interpolation]
COCOS evidence: EquilibriumConvention(cocos=11, candidates=(11,), psi_per_radian=False, clockwise_phi=None, ip_sign=1, bt_sign=1, q_sign=1, source='argument', identified=(1, 2))


In [ ]:
coordinates = geq_desc.radial_coordinates
print("Coordinate definitions:")
for name, item in coordinates.items():
    print(f"- {name}: {item.definition}; available={item.available}")
print("Validation diagnostics:", [issue.message for issue in geq_desc.validation.issues])

Coordinate definitions:
- psi_n: (psi-psi_axis)/(psi_boundary-psi_axis); available=True
- rho_pol_n: sqrt(psi_n); available=True
- rho_tor_n: sqrt(int_axis^psi q dpsi / int_axis^boundary q dpsi); available=True
Validation diagnostics: ['COCOS 11 is declared (argument) but the observable signs and flux scale support (1, 2); the declaration is being used as given']


## Current-distribution moments

The descriptors above describe the boundary and the profiles. A current
moment describes the toroidal **current density** itself, in a hierarchy
between a single filament and the full $J_\phi(R, Z)$:

$$I_p = \int J_\phi\,dA,\qquad (R_c, Z_c) = \frac{1}{I_p}\int (R, Z)\,J_\phi\,dA,\qquad
\mu_{pq} = \frac{1}{I_p}\int (R-R_c)^p (Z-Z_c)^q\,J_\phi\,dA.$$

`derive_current_moments` evaluates $J_\phi = -\sigma_{B_p}(2\pi)^{e_{B_p}}(R\,p' + FF'/\mu_0 R)$
from the record's own flux functions in its declared COCOS (or takes a
supplied `j_tor`). It integrates over the LCFS with fractional cell weights.
The covariance $[[\mu_{20}, \mu_{11}], [\mu_{11}, \mu_{02}]]$ gives the widths
and tilt of the *current*, and the odd moments its skewness. Neither is the
boundary's minor radius, elongation or triangularity.

In [ ]:
from vaft.process.equilibrium import derive_current_moments

# The current density depends on the flux unit, so this uses the g-file's own
# identified convention (COCOS 1 or 2, flux per radian) rather than the COCOS 11
# declared above for the descriptor comparison. Declaring 11 on this file makes
# J_phi 2*pi too large, and derive_current_moments warns that its I_p then
# disagrees with the file's.
geq_native = as_equilibrium(gfile)
moments = derive_current_moments(geq_native, max_order=3)
print(f"current density from : {moments.current_density_source}")
print(f"I_p                  : {moments.total_current:.6g} A   (g-file CURRENT {geq_native.ip:.6g} A)")
print(f"centroid (R_c, Z_c)  : ({moments.centroid_r:.4f}, {moments.centroid_z:.4f}) m   "
      f"magnetic axis ({geq_native.magnetic_axis[0]:.4f}, {geq_native.magnetic_axis[1]:.4f}) m")
widths = np.sqrt(np.linalg.eigvalsh(moments.covariance))
print(f"principal current widths : {widths[0]:.4f}, {widths[1]:.4f} m  (current-width ratio {widths[1]/widths[0]:.3f})")
for key in ((3, 0), (2, 1), (1, 2), (0, 3)):
    print(f"mu_{key[0]}{key[1]} = {moments.central_moments[key]: .3e} m^3")


current density from : grad_shafranov_flux_functions
I_p                  : 77032.3 A   (g-file CURRENT 77040.1 A)
centroid (R_c, Z_c)  : (0.3989, 0.0000) m   magnetic axis (0.4813, 0.0000) m
principal current widths : 0.1448, 0.1821 m  (current-width ratio 1.258)
mu_30 =  6.185e-04 m^3
mu_21 = -1.214e-09 m^3
mu_12 = -7.499e-04 m^3
mu_03 =  2.906e-09 m^3


The same reduction on the analytic Cerfon–Freidberg family shows what the
odd moments measure. The limited and double-null cases are up-down symmetric,
so their odd-$Z$ moments vanish; the lower single null's do not. A single
filament would carry the same $I_p$ and centroid with every higher central
moment zero.

In [ ]:
from vaft.process.equilibrium import solovev_example

print(f"{'topology':12s} {'I_p [kA]':>9s} {'R_c':>7s} {'Z_c':>8s} {'mu_20':>9s} {'mu_02':>9s} {'mu_03':>10s}")
for topology in ("limited", "double_null", "single_null"):
    m = derive_current_moments(solovev_example(topology), max_order=3)
    mu = m.central_moments
    print(f"{topology:12s} {m.total_current/1e3:9.2f} {m.centroid_r:7.4f} {m.centroid_z:8.4f} "
          f"{mu[(2, 0)]:9.2e} {mu[(0, 2)]:9.2e} {mu[(0, 3)]:10.2e}")


topology      I_p [kA]     R_c      Z_c     mu_20     mu_02      mu_03


limited          99.92  0.3469  -0.0000  1.37e-02  4.00e-02  -1.38e-18


double_null      99.89  0.3432   0.0000  1.29e-02  3.63e-02   2.22e-19


single_null      99.95  0.3454   0.0175  1.34e-02  3.80e-02  -5.75e-04
